# YouTube + Google Meet — 3 Mbps / 100 ms / pfifo

## PCAP-only partial report

The interrupted 3 Mbps run produced a PCAP but no YouTube or Google Meet QoE JSONL files. Traffic results below use the busiest 15-second inbound window in the capture. Application attribution includes the explicitly requested assumptions; playback-health, resolution, jitter, and frame-delivery QoE cannot be recovered from the PCAP.

In [ ]:
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

BANDWIDTH_MBPS = 3
CLIENT_IP = '172.16.1.1'
WINDOW_SECONDS = 15
result_candidates = [
    Path('youtube_google_meet_100ms_pfifo/results/3mbps'),
    Path('experiments/youtube_google_meet_100ms_pfifo/results/3mbps'),
]
RESULT_DIR = next((p.resolve() for p in result_candidates if p.is_dir()), None)
if RESULT_DIR is None:
    raise FileNotFoundError('Could not locate the 3 Mbps result directory')
PCAP = next((p for p in RESULT_DIR.glob('*.pcap') if not p.name.endswith('.partial')), None)
if PCAP is None:
    raise FileNotFoundError(f'No PCAP found in {RESULT_DIR}')
plt.style.use('seaborn-v0_8-whitegrid')

sni_output = subprocess.run(
    ['tshark', '-r', str(PCAP), '-Y', 'tls.handshake.extensions_server_name',
     '-T', 'fields', '-e', 'ip.dst', '-e', 'tls.handshake.extensions_server_name'],
    check=True, capture_output=True, text=True,
).stdout
hosts_by_ip = {}
for line in sni_output.splitlines():
    fields = line.split('\t')
    if len(fields) >= 2 and fields[0] and fields[1]:
        hosts_by_ip.setdefault(fields[0], set()).update(h.lower() for h in fields[1].split(','))
youtube_markers = ('youtube.com', 'googlevideo.com', 'ytimg.com', 'ggpht.com', 'gvt1.com')
meet_markers = ('meet.google.com', 'hangouts.clients6.google.com')
youtube_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in youtube_markers)}
meet_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in meet_markers)} - youtube_ips

packet_output = subprocess.run(
    ['tshark', '-r', str(PCAP), '-Y', f'ip.dst == {CLIENT_IP}', '-T', 'fields',
     '-e', 'frame.time_epoch', '-e', 'ip.src', '-e', 'udp.srcport', '-e', 'tcp.srcport',
     '-e', 'frame.len', '-e', '_ws.col.Protocol'],
    check=True, capture_output=True, text=True,
).stdout
rows = []
for line in packet_output.splitlines():
    fields = line.split('\t')
    if len(fields) < 6 or not fields[0] or not fields[1] or not fields[4]:
        continue
    timestamp = float(fields[0])
    remote_ip = fields[1]
    remote_port = int((fields[2] or fields[3] or '0').split(',')[0])
    frame_bytes = int(fields[4].split(',')[0])
    protocol = fields[5] or 'Unknown'
    if remote_ip in youtube_ips:
        application = 'YouTube'
    elif remote_ip in meet_ips or 19302 <= remote_port <= 19309:
        application = 'Google Meet'
    else:
        application = 'Unclassified'
    rows.append((timestamp, remote_ip, remote_port, protocol, frame_bytes, application))

all_packets = pd.DataFrame(rows, columns=['timestamp', 'remote_ip', 'remote_port', 'protocol', 'frame_bytes', 'application'])
if all_packets.empty:
    raise RuntimeError('No inbound IPv4 packets found in the PCAP')
capture_start = all_packets.timestamp.min()
all_packets['capture_second'] = (all_packets.timestamp - capture_start).astype(int)
per_second = all_packets.groupby('capture_second').frame_bytes.sum().reindex(
    range(all_packets.capture_second.max() + 1), fill_value=0
)
rolling = per_second.rolling(WINDOW_SECONDS, min_periods=1).sum()
end_bin = int(rolling.idxmax())
start_bin = max(0, end_bin - WINDOW_SECONDS + 1)
window_start = capture_start + start_bin
window_end = window_start + WINDOW_SECONDS
packets = all_packets[(all_packets.timestamp >= window_start) & (all_packets.timestamp < window_end)].copy()

assumption_candidates = (packets[
    (packets.application == 'Unclassified') & packets.remote_ip.str.startswith('172.217.')
].groupby('remote_ip').frame_bytes.sum().sort_values(ascending=False).head(2))
assumed_ip_map = {}
if len(assumption_candidates) >= 1:
    assumed_ip_map[assumption_candidates.index[0]] = 'Google Meet'
if len(assumption_candidates) >= 2:
    assumed_ip_map[assumption_candidates.index[1]] = 'YouTube'
assumed_rows = []
for remote_ip, application in assumed_ip_map.items():
    assumed_rows.append({
        'remote IP': remote_ip,
        'assumed application': application,
        'inbound download (MiB)': assumption_candidates[remote_ip] / 2**20,
    })
    packets.loc[packets.remote_ip == remote_ip, 'application'] = application
remaining_meet_bytes = packets.loc[packets.application == 'Unclassified', 'frame_bytes'].sum()
packets.loc[packets.application == 'Unclassified', 'application'] = 'Google Meet'

print(f'PCAP: {PCAP.name}')
print(f'Capture duration represented: {all_packets.timestamp.max() - all_packets.timestamp.min():.3f} seconds')
print(f'Selected busiest traffic window: capture seconds {start_bin}–{start_bin + WINDOW_SECONDS}')
print('Requested heuristic attribution (assumption, not packet-level ground truth):')
if assumed_rows:
    display(pd.DataFrame(assumed_rows).set_index('remote IP').round(3))
print(f'Other previously-unclassified inbound traffic assigned to Google Meet: {remaining_meet_bytes / 2**20:.3f} MiB')

## Traffic during the busiest 15-second capture window

In [ ]:
packets['second'] = (packets.timestamp - window_start).astype(int)
applications = ['YouTube', 'Google Meet']
app_bytes = (packets.groupby(['second', 'application']).frame_bytes.sum()
             .unstack(fill_value=0).reindex(range(WINDOW_SECONDS), fill_value=0)
             .reindex(columns=applications, fill_value=0))
app_mbps = app_bytes * 8 / 1_000_000
total_mbps = app_mbps.sum(axis=1)
attributed_bytes = app_bytes.to_numpy().sum()
summary_rows = []
for application in applications:
    total_bytes = app_bytes[application].sum()
    summary_rows.append({
        'application': application,
        'download (MiB)': total_bytes / 2**20,
        'average in selected window (Mbps)': total_bytes * 8 / WINDOW_SECONDS / 1_000_000,
        'peak 1-second bin (Mbps)': app_mbps[application].max(),
        'share of assumed bytes (%)': 100 * total_bytes / attributed_bytes if attributed_bytes else 0,
    })
display(pd.DataFrame(summary_rows).set_index('application').round(3))

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
ax.plot(app_mbps.index, app_mbps['YouTube'], marker='o', color='#ff0000', label='YouTube (identified + assumed)')
ax.plot(app_mbps.index, app_mbps['Google Meet'], marker='o', color='#1f78b4', label='Google Meet (identified + assumed)')
ax.plot(total_mbps.index, total_mbps, marker='.', color='#111111', linewidth=2, label='Total inbound')
ax.axhline(BANDWIDTH_MBPS, color='#555555', linestyle='--', alpha=.8, label='Configured bottleneck (3 Mbps)')
ax.set(title='Inbound traffic during busiest 15-second PCAP window', xlabel='Seconds from selected window start', ylabel='Downloaded Mbit in each 1-second bin')
ax.set_xticks(range(WINDOW_SECONDS))
ax.legend(loc='best')
plt.tight_layout()
plt.show()
print(f'Total inbound download: {attributed_bytes / 2**20:.3f} MiB')
print(f'Average total inbound rate: {attributed_bytes * 8 / WINDOW_SECONDS / 1_000_000:.3f} Mbps')

## Protocol composition and largest inbound endpoints

In [ ]:
protocol_bytes = packets.groupby('protocol').frame_bytes.sum().sort_values(ascending=False)
protocol_mib = (protocol_bytes / 2**20).head(10)
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
protocol_mib.plot(kind='bar', ax=ax, color='#4c78a8')
ax.set(title='Inbound traffic by decoded protocol', xlabel='Wireshark protocol', ylabel='Download (MiB)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

top_endpoints = (packets.groupby(['application', 'remote_ip', 'remote_port', 'protocol']).frame_bytes.sum()
                 .sort_values(ascending=False).head(15).rename('download bytes').reset_index())
top_endpoints['download (MiB)'] = top_endpoints.pop('download bytes') / 2**20
display(top_endpoints.round(3))

## QoE metrics unavailable

The 3 Mbps run has no `youtube_stats.jsonl` or `google_meet_stats.jsonl`. Consequently, buffer health, dropped frames, video resolution, Meet bitrate/jitter, decoded frames, and freezes cannot be plotted from this artifact.